In [1]:
# Test script 2

In [1]:
# Calculate global mortality - requires ~90GB memory

In [2]:
import os
import xarray as xr
import numpy as np
import warnings
from utils.utils import get_scenario_config
from utils.mortality_utils import att_frac

In [3]:
# === Path config ===
MASKS_DIR = "/glade/work/awells/air_quality/BMR/masks/country/"
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
BMR_DIR = "/glade/derecho/scratch/awells/air_quality/BMR/"

In [4]:
# Load country masks
mask_file = "GBD_Country_Masks_0.10.nc"
mask_path = os.path.join(MASKS_DIR, mask_file)
masks = xr.open_dataarray(mask_path)

# Load population file
pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
population = xr.open_dataarray(pop_path)
pop = population.reindex_like(masks, method="nearest", tolerance=1e-9)

In [5]:
# === Calculate the scalar distributions ===
n_samples = 1000

# TMREL from GBD21 (uniform distribution)
tmrel_low = 29.1
tmrel_high = 35.7
tmrel_samples = np.random.uniform(tmrel_low, tmrel_high, size=n_samples)

tmrel_da = xr.DataArray(
    tmrel_samples,
    dims=['samples'],
    coords={'samples': np.arange(n_samples)}
).astype("float32")

# Beta from RR per 10ppb (normal distribution)
RR_10 = 1.074
RR_10_lower = 1.014
RR_10_upper = 1.137
beta_mean = np.log(RR_10) / 10
beta_std = (np.log(RR_10_upper) - np.log(RR_10_lower)) / (2 * 1.96 * 10)
beta_samples = np.random.normal(beta_mean, beta_std, size=n_samples)

beta_da = xr.DataArray(
    beta_samples,
    dims=['samples'],
    coords={'samples': np.arange(n_samples)}
).astype("float32")

# Load BMR for each grid point
bmr_file = f"GBD_BMR_Country_Mask_COPD_{n_samples}_samples_1990-2009.nc"
bmr_path = os.path.join(BMR_DIR, bmr_file)
BMR = xr.open_dataarray(bmr_path)  # three quantiles

In [ ]:
warnings.filterwarnings('ignore')

# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "G6-1.5K"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

O3_DIR = f"/glade/work/awells/air_quality/{model}/ozone/OSDMA8_BC/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/mortality/ozone/global/"

for ens_num in ensemble_members:
    print(f"Processing ensemble member {ens_num:02d}")
    dates = f"{years.start}-{years.stop - 1}"

    # Load ozone data
    o3_file = f"OSDMA8_BC_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    o3_path = os.path.join(O3_DIR, o3_file)
    o3 = xr.open_dataarray(o3_path).astype("float32")
    o3 = o3.reindex_like(masks, method="nearest", tolerance=1e-9, fill_value=0)

    del o3_file, o3_path

    # make o3 dask-backed
    o3 = o3.chunk({'lat': 180, 'lon': 360})

    # range doesn't include the final year which is okay
    # because OSDMA8 excludes final year
    for year in years:
        print(f"Processing year {year}")
        o3_year = o3.sel(year=year)
        AF = att_frac(o3_year, tmrel_da, beta_da).chunk({"samples": 10})

        POP = pop.sel(year=year).chunk({"lat": 180, "lon": 360})
        M = AF * BMR * POP
        global_M = M.sum(dim=("lat", "lon"))

        del o3_year, AF, POP, M

        # mean over samples
        M_mean = global_M.mean(dim="samples")
        # median over samples
        M_median = global_M.quantile(0.5, dim="samples")
        # quantiles for box plotting
        M_25 = global_M.quantile(0.25, dim="samples")
        M_75 = global_M.quantile(0.75, dim="samples")
        # percentiles for 95% CI
        M_lower = global_M.quantile(0.025, dim="samples")
        M_upper = global_M.quantile(0.975, dim="samples")

        del global_M

        M_summary = xr.Dataset({
            "M_mean": M_mean.astype("float32").drop_vars("year"),
            "M_median": M_median.astype("float32").drop_vars("quantile"),
            "M_25": M_25.astype("float32").drop_vars("quantile"),
            "M_75": M_75.astype("float32").drop_vars("quantile"),
            "M_lower95": M_lower.astype("float32").drop_vars("quantile"),
            "M_upper95": M_upper.astype("float32").drop_vars("quantile")
        })

        del M_mean, M_median, M_25, M_75, M_lower, M_upper

        description = ("Global mortality (COPD) due to ozone "
                       "statistics: including mean, median, "
                       "and the 95% CI - scripts by A.F. Wells (2025)")
        M_summary.attrs["description"] = description
        M_summary.attrs["model"] = model
        M_summary.attrs["scenario"] = scenario
        M_summary.attrs["ensemble_number"] = ens_num
        M_summary.attrs["year"] = year

        out_file = f"Global_mortality_stats_{model}_{scenario}_{ens_num:02d}_{year}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)
        print(f"Saving to {out_path}")
        M_summary.to_netcdf(out_path)

        del M_summary

    del o3

print("All processing complete.")

Processing ensemble member 01
Processing year 2035
Saving to /glade/work/awells/air_quality/CESM2/mortality/ozone/global/Global_mortality_stats_CESM2_G6-1.5K_01_2035.nc
Processing year 2036
Saving to /glade/work/awells/air_quality/CESM2/mortality/ozone/global/Global_mortality_stats_CESM2_G6-1.5K_01_2036.nc
Processing year 2037
Saving to /glade/work/awells/air_quality/CESM2/mortality/ozone/global/Global_mortality_stats_CESM2_G6-1.5K_01_2037.nc
Processing year 2038
Saving to /glade/work/awells/air_quality/CESM2/mortality/ozone/global/Global_mortality_stats_CESM2_G6-1.5K_01_2038.nc
Processing year 2039
Saving to /glade/work/awells/air_quality/CESM2/mortality/ozone/global/Global_mortality_stats_CESM2_G6-1.5K_01_2039.nc
Processing year 2040
Saving to /glade/work/awells/air_quality/CESM2/mortality/ozone/global/Global_mortality_stats_CESM2_G6-1.5K_01_2040.nc
Processing year 2041
Saving to /glade/work/awells/air_quality/CESM2/mortality/ozone/global/Global_mortality_stats_CESM2_G6-1.5K_01_2041.